# MindTrack · Sistema Inteligente para la Predicción del Riesgo Académico
## Entregable 2 — Validación, Exploración, Preprocesamiento y Partición de Datos

**Autores:** Geraldin Perez · Daniel Hernandez

Este notebook implementa, paso a paso, el flujo descrito en la presentación técnica de MindTrack:

| Bloque | Contenido | Sección del PDF |
|---|---|---|
| 1 | Configuración e importación de librerías | — |
| 2 | Carga, descripción y validación de calidad | 01 |
| 3 | Análisis Exploratorio de Datos (EDA) | 02 – 07 |
| 4 | Pipeline de preprocesamiento | 08 |
| 5 | Partición estratificada 70 / 15 / 15 | 09 |
| 6 | Métricas de evaluación y baseline (Regresión Logística) | 10 · Modelos |

**Variable objetivo:** `Academic_Failure_Risk` (1 = estudiante en riesgo, 0 = sin riesgo).
**Criterio de éxito:** `Recall (Clase 1) ≥ 0.85` → minimizar los Falsos Negativos (estudiantes en riesgo no detectados).

---
### Mapa de notebooks · Proyecto MindTrack

Los entregables de modelado se dividen en **dos Colabs independientes** (no se sustituyen entre sí):

| Entregable | Archivo | Qué contiene |
|:---:|:---|:---|
| **2** | `MindTrack_EDA_Preprocesamiento.ipynb` | Calidad, EDA (diapositivas), preprocesamiento, split 70/15/15 y **línea base** (solo Regresión Logística) |
| **3** | `MindTrack_03_Comparacion_Modelos.ipynb` | **Comparación** Regresión Logística · Random Forest · XGBoost, gráficas y guardado en Drive |

**Usted está en el Entregable 2.**  
Para los tres modelos y la selección del ganador, abra el [Notebook 3 en Colab](https://colab.research.google.com/gist/Danielhsk/92fab1e431edaeb0ce998bb03a09d04c/MindTrack_03_Comparacion_Modelos.ipynb).

Constantes compartidas: `random_state=42`, sin `Academic_Performance_Score` (anti-leakage), `Recall (C1) ≥ 0.85`.


[![Abrir Entregable 2 en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/gist/Danielhsk/8c4e3e85e3efde9c60b9db408a507afb/MindTrack_EDA_Preprocesamiento.ipynb)

> **Solo EDA + baseline (1 modelo).** La comparación de 3 algoritmos está en el [Entregable 3](https://colab.research.google.com/gist/Danielhsk/92fab1e431edaeb0ce998bb03a09d04c/MindTrack_03_Comparacion_Modelos.ipynb).

**Dataset:** `Sistemas inteligentes / DataSet / academic_risk_dataset.csv` en Drive, o en `/content`.


---
## 1. Configuración e importación de librerías

Se importan las librerías de manipulación de datos, visualización y modelado, y se define una
identidad visual sobria (azul marino / slate) alineada con el diseño corporativo de MindTrack.

In [ ]:
# --- Librerías estándar ---
import warnings
from pathlib import Path

# --- Manipulación de datos ---
import numpy as np
import pandas as pd

# --- Visualización ---
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# --- Modelado y evaluación (scikit-learn) ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.3f}".format)

### 1.1 Constantes globales del proyecto
Centralizar rutas, nombres de columnas y semillas garantiza **reproducibilidad** y facilita el mantenimiento.

In [ ]:
# Reproducibilidad
RANDOM_STATE = 42

# Datos
DATA_FILENAME = "academic_risk_dataset.csv"
# Opcional: ruta fija en Google Drive (None = buscar en /content)
DRIVE_DATA_PATH = "/content/drive/MyDrive/Sistemas inteligentes/DataSet/academic_risk_dataset.csv"  # None si solo usa /content
DATA_PATH = Path(DATA_FILENAME)  # se reasigna en la celda de entorno
EXPECTED_ROWS, EXPECTED_COLS = 15_000, 14

# Variables clave
TARGET = "Academic_Failure_Risk"
ID_COLS = ["Student_ID"]
LEAKAGE_COLS = ["Academic_Performance_Score"]   # Fuga de información (data leakage)
NOMINAL_COLS = ["Gender", "Education_Level"]    # One-Hot Encoding
ORDINAL_COL = "Burnout_Level"                   # Mapeo ordinal 0-3
BURNOUT_ORDER = ["Low", "Moderate", "High", "Severe"]
BURNOUT_MAP = {level: code for code, level in enumerate(BURNOUT_ORDER)}

CONTINUOUS_COLS = [
    "Daily_Social_Media_Hours",
    "Daily_AI_Tool_Usage_Hours",
    "Sleep_Hours",
    "Mental_Health_Score",
    "Academic_Performance_Score",
    "Social_Isolation_Score",
]

# Preprocesamiento y evaluación
DROP_FIRST_DUMMY = True        # Evita la trampa de variables ficticias (colinealidad)
EXPECTED_COLS_AFTER_DROP = 12  # 14 - Student_ID - Academic_Performance_Score (cifra del PDF)
EXPECTED_N_FEATURES = 13       # 9 numéricas/ordinales + 2 dummies Gender + 2 dummies Education_Level
RECALL_TARGET = 0.85           # Criterio de éxito (Sección 10)

In [ ]:
# Entorno Google Colab / Jupyter local
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess

    subprocess.check_call(
        ["pip", "install", "-q", "scikit-learn", "seaborn", "matplotlib", "pandas"]
    )


def locate_dataset(filename: str = DATA_FILENAME) -> Path | None:
    # Busca el CSV sin pedir subida si ya esta en Colab, Drive o el directorio local.
    candidates: list[Path] = []

    if DRIVE_DATA_PATH:
        candidates.append(Path(DRIVE_DATA_PATH))

    for root in (Path("/content"), Path.cwd()):
        candidates.append(root / filename)

    candidates.append(Path("/content/drive/MyDrive") / filename)

    seen: set[str] = set()
    for raw in candidates:
        try:
            path = raw.expanduser()
            key = str(path)
        except OSError:
            continue
        if key in seen:
            continue
        seen.add(key)
        if path.is_file():
            return path.resolve()

    content = Path("/content")
    if content.is_dir():
        hits = sorted(content.rglob(filename))
        if hits:
            return hits[0].resolve()
    return None


def prompt_upload(filename: str = DATA_FILENAME) -> Path | None:
    from google.colab import files

    print(f"No se encontró {filename} automáticamente. Suba el archivo ahora:")
    uploaded = files.upload()
    if not uploaded:
        return None
    name = next(iter(uploaded))
    target = Path("/content") / filename
    src = Path(name)
    if not src.is_file():
        src = Path("/content") / name
    if src.is_file() and src.resolve() != target.resolve():
        if target.exists():
            target.unlink()
        src.replace(target)
    return target if target.is_file() else None


if IN_COLAB and DRIVE_DATA_PATH and not Path(DRIVE_DATA_PATH).is_file():
    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.is_dir():
        from google.colab import drive

        print("Montando Google Drive para ubicar el dataset...")
        drive.mount("/content/drive", force_remount=False)

found = locate_dataset()
if found is None and IN_COLAB:
    found = prompt_upload()
elif found is None and not IN_COLAB:
    local = Path(DATA_FILENAME)
    if local.is_file():
        found = local.resolve()

if found is None:
    raise FileNotFoundError(
        f"No se encontró '{DATA_FILENAME}'. En Colab, deje el CSV en /content "
        f"(panel Archivos), guárdelo en Drive y defina DRIVE_DATA_PATH, o súbalo en esta celda."
    )

DATA_PATH = found
if IN_COLAB:
    print(f"Colab · dataset listo: {DATA_PATH}")
else:
    print(f"Local · dataset listo: {DATA_PATH.resolve()}")

### 1.2 Identidad visual MindTrack
Paleta corporativa: azul marino como color principal, gamas *slate* como secundarios y un ámbar
apagado reservado **exclusivamente** para resaltar la clase en riesgo.

In [ ]:
# Paleta corporativa MindTrack
MT_NAVY = "#1F3A5F"
MT_SLATE = "#475569"
MT_STEEL = "#64748B"
MT_MIST = "#94A3B8"
MT_LIGHT = "#CBD5E1"
MT_INK = "#0F172A"
MT_ACCENT = "#B45309"   # Resalta la clase en riesgo (Clase 1)

MT_PALETTE = [MT_NAVY, MT_SLATE, MT_STEEL, MT_MIST, MT_LIGHT, MT_INK]
MT_RISK_PALETTE = {0: MT_NAVY, 1: MT_ACCENT}
MT_CMAP = LinearSegmentedColormap.from_list("mindtrack", [MT_ACCENT, "#F8FAFC", MT_NAVY])

sns.set_theme(
    style="whitegrid",
    palette=MT_PALETTE,
    rc={
        "figure.figsize": (10, 5),
        "figure.dpi": 100,
        "axes.edgecolor": MT_LIGHT,
        "axes.labelcolor": MT_INK,
        "axes.titlecolor": MT_NAVY,
        "axes.titleweight": "bold",
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "grid.color": "#E2E8F0",
        "xtick.color": MT_SLATE,
        "ytick.color": MT_SLATE,
        "font.family": "sans-serif",
    },
)


def print_header(title: str, width: int = 72) -> None:
    """Imprime un encabezado con formato uniforme en consola."""
    line = "=" * width
    print(f"\n{line}\n{title.upper().center(width)}\n{line}")


def check(condition: bool, message: str) -> None:
    """Imprime una verificación con estado OK / ALERTA."""
    status = "[OK]    " if condition else "[ALERTA]"
    print(f"{status} {message}")


def style_axis(ax, title: str, xlabel: str = "", ylabel: str = "") -> None:
    """Aplica el estilo MindTrack a un eje."""
    ax.set_title(title, loc="left")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    sns.despine(ax=ax, left=True)


print_header("Entorno MindTrack configurado")
print(f"pandas {pd.__version__} · numpy {np.__version__} · seaborn {sns.__version__}")
sns.palplot(MT_PALETTE + [MT_ACCENT])
plt.title("Paleta MindTrack", loc="left", color=MT_NAVY, fontweight="bold")
plt.show()

In [ ]:
# --- Constantes y estilo de gráficas (idénticas a la presentación Entregable 2) ---
# Paleta diapositivas: verde = sin riesgo, rojo = en riesgo (slides 4, 10, 14)
SLIDE_RISK_PALETTE = {0: "#2ca02c", 1: "#d62728"}

# Variables del EDA univariado en las diapositivas 6, 9 y 12 (rejilla 2×3)
HABIT_EDA_COLS = [
    "Daily_Social_Media_Hours",
    "Daily_AI_Tool_Usage_Hours",
    "Sleep_Hours",
    "Mental_Health_Score",
    "Academic_Performance_Score",
    "Social_Isolation_Score",
]

# Slide 8 — diferencias de medias
COMPARE_MEAN_COLS = [
    "Sleep_Hours",
    "Daily_Social_Media_Hours",
    "Mental_Health_Score",
]
COMPARE_MEAN_LABELS = {
    "Sleep_Hours": "Sueño (h/día)",
    "Daily_Social_Media_Hours": "Redes Sociales (h/día)",
    "Mental_Health_Score": "Salud Mental (pts)",
}

# Slide 14 — boxplots por riesgo
COMPARE_RISK_COLS = [
    "Daily_Social_Media_Hours",
    "Sleep_Hours",
    "Mental_Health_Score",
]
COMPARE_RISK_TITLES = {
    "Daily_Social_Media_Hours": "Redes Sociales por Riesgo",
    "Sleep_Hours": "Horas de Sueño por Riesgo",
    "Mental_Health_Score": "Salud Mental por Riesgo",
}

# Medias globales de referencia (slide 6)
SLIDE_MEANS_REF = {
    "Daily_Social_Media_Hours": (4.54, 2.40),
    "Daily_AI_Tool_Usage_Hours": (2.58, 1.68),
    "Sleep_Hours": (6.54, 1.27),
    "Physical_Activity_Hours": (1.25, 1.04),
}


def plot_target_distribution_slide(data: pd.DataFrame, target: str) -> None:
    """Slide 4 — distribución de la variable objetivo."""
    counts = data[target].value_counts().sort_index()
    labels = ["Sin Riesgo (0)", "En Riesgo (1)"]
    colors = [SLIDE_RISK_PALETTE[i] for i in counts.index]
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(labels, counts.values, color=colors, edgecolor="black", linewidth=0.6)
    total = len(data)
    for bar, n in zip(bars, counts.values):
        pct = n / total * 100
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 150,
            f"{n:,.1f}\n({pct:.1f}%)".replace(",", "."),
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )
    ax.set_ylabel("Cantidad de Estudiantes")
    ax.set_xlabel("Riesgo Académico (0: Sin Riesgo, 1: En Riesgo)")
    ax.set_title("Distribución de la Variable Objetivo (Academic_Failure_Risk)", fontweight="bold")
    ax.set_ylim(0, counts.max() * 1.12)
    plt.tight_layout()
    plt.show()


def plot_means_by_risk_slide(data: pd.DataFrame, columns: list, target: str) -> None:
    """Slide 8 — diferencias de medias entre clases."""
    means = data.groupby(target)[columns].mean()
    x = np.arange(len(columns))
    width = 0.35
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.bar(
        x - width / 2,
        means.loc[0].values,
        width,
        label="Sin Riesgo",
        color=SLIDE_RISK_PALETTE[0],
        edgecolor="black",
        linewidth=0.5,
    )
    ax.bar(
        x + width / 2,
        means.loc[1].values,
        width,
        label="En Riesgo",
        color=SLIDE_RISK_PALETTE[1],
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_xticks(x)
    ax.set_xticklabels([COMPARE_MEAN_LABELS.get(c, c) for c in columns])
    ax.set_ylabel("Media")
    ax.set_title("Diferencias de Medias: En Riesgo vs Sin Riesgo", fontweight="bold")
    ax.legend()
    for i, col in enumerate(columns):
        v0, v1 = means.loc[0, col], means.loc[1, col]
        ax.text(i - width / 2, v0 + 1, f"{v0:.2f}", ha="center", fontsize=9)
        ax.text(i + width / 2, v1 + 1, f"{v1:.2f}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()


def plot_burnout_risk_percent_slide(data: pd.DataFrame, ordinal_col: str, target: str,
                                    order: list) -> None:
    """Slide 10 — % de riesgo por nivel de burnout (barras apiladas 100 %)."""
    pct = (
        data.groupby(ordinal_col)[target]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .reindex(order)
        .sort_index(axis=1)
    )
    pct = pct * 100
    fig, ax = plt.subplots(figsize=(8, 5))
    bottom = np.zeros(len(order))
    for cls in sorted(pct.columns):
        vals = pct[cls].values
        color = SLIDE_RISK_PALETTE.get(cls, MT_SLATE)
        label = "Sin Riesgo (0)" if cls == 0 else "En Riesgo (1)"
        bars = ax.bar(order, vals, bottom=bottom, color=color, label=label, edgecolor="white", linewidth=0.8)
        for bar, v, b in zip(bars, vals, bottom):
            if v > 4:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    b + v / 2,
                    f"{v:.1f}%",
                    ha="center",
                    va="center",
                    color="white",
                    fontweight="bold",
                    fontsize=10,
                )
        bottom += vals
    ax.set_ylabel("Porcentaje (%)")
    ax.set_xlabel("Nivel de Burnout")
    ax.set_title("Porcentaje de Riesgo Académico por Nivel de Burnout", fontweight="bold")
    ax.set_ylim(0, 100)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


def plot_categorical_slide(data: pd.DataFrame, col: str, palette: str, title: str) -> None:
    """Slides 11 — barras categóricas con paletas secuenciales."""
    order = data[col].value_counts().index.tolist()
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.countplot(data=data, x=col, order=order, palette=palette, ax=ax, edgecolor="black", linewidth=0.4)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Cantidad")
    ax.tick_params(axis="x", rotation=15)
    plt.tight_layout()
    plt.show()


def plot_histograms_slide(data: pd.DataFrame, columns: list) -> None:
    """Slide 9 — histogramas + KDE (rejilla 2×3)."""
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, col in zip(axes.flat, columns):
        sns.histplot(data[col], bins=30, kde=True, ax=ax, color="#5b9bd5", edgecolor="white")
        ax.set_title(f"Histograma: {col}", fontweight="bold")
        ax.set_ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()


def plot_boxplots_slide(data: pd.DataFrame, columns: list) -> None:
    """Slide 12 — diagramas de caja horizontales naranja."""
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    for ax, col in zip(axes.flat, columns):
        sns.boxplot(x=data[col], ax=ax, color="orange", width=0.55)
        ax.set_title(f"Diagrama de Caja: {col}", fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_correlation_slide(data: pd.DataFrame, id_cols: list, ordinal_col: str,
                         burnout_map: dict, target: str) -> pd.DataFrame:
    """Slide 13 — matriz de correlación (coolwarm, anotación .2f)."""
    corr_df = data.drop(columns=[c for c in id_cols if c in data.columns]).copy()
    corr_df[ordinal_col] = corr_df[ordinal_col].astype(str).str.strip().str.title().map(burnout_map)
    corr_df = corr_df.select_dtypes(include="number")
    corr_matrix = corr_df.corr(method="pearson")
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(
        corr_matrix,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        center=0,
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.5,
        annot_kws={"size": 8},
        ax=ax,
    )
    ax.set_title("Matriz de Correlación de Pearson", fontweight="bold")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
    return corr_matrix


def plot_risk_boxplots_slide(data: pd.DataFrame, columns: list, target: str,
                             titles: dict) -> None:
    """Slide 14 — boxplots por nivel de riesgo."""
    plot_df = data.copy()
    plot_df["_risk_label"] = plot_df[target].map({0: "Sin Riesgo (0)", 1: "En Riesgo (1)"})
    fig, axes = plt.subplots(1, len(columns), figsize=(5 * len(columns), 5))
    palette = {"Sin Riesgo (0)": SLIDE_RISK_PALETTE[0], "En Riesgo (1)": SLIDE_RISK_PALETTE[1]}
    for ax, col in zip(np.atleast_1d(axes), columns):
        sns.boxplot(
            data=plot_df,
            x="_risk_label",
            y=col,
            order=["Sin Riesgo (0)", "En Riesgo (1)"],
            palette=palette,
            ax=ax,
            width=0.55,
        )
        ax.set_title(titles.get(col, col), fontweight="bold")
        ax.set_xlabel("Academic_Failure_Risk")
        ax.set_ylabel(col)
    plt.tight_layout()
    plt.show()


---
## 2. Carga y descripción de los datos · *Sección 01 — Calidad y estructura*

**Hallazgos esperados (PDF):** 15.000 registros de Kaggle, 14 variables (13 predictoras/descriptivas +
`Academic_Failure_Risk`), **0 % duplicados** y **0 % faltantes** → no se requieren imputaciones ni
deduplicación, preservando la trazabilidad completa.

In [ ]:
# Carga del dataset
df = pd.read_csv(DATA_PATH)

print_header("Dimensiones del dataset")
print(f"Registros : {df.shape[0]:>7,}".replace(",", "."))
print(f"Variables : {df.shape[1]:>7}")
check(df.shape == (EXPECTED_ROWS, EXPECTED_COLS),
      f"Forma esperada {EXPECTED_ROWS:,} x {EXPECTED_COLS}".replace(",", "."))

### 2.1 Primeros registros

In [ ]:
df.head(10)

### 2.2 Estructura y tipos de datos (`.info()`)

In [ ]:
df.info()

### 2.3 Estadística descriptiva — variables numéricas

In [ ]:
df.describe().T

### 2.4 Estadística descriptiva — variables categóricas

In [ ]:
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
df[categorical_cols].describe().T

### 2.5 Validación de calidad: nulos, duplicados y rangos
Se confirma la ausencia de valores faltantes y registros duplicados (tanto filas completas como
`Student_ID` repetidos), y que los rangos de horas sean físicamente posibles (0–24 h).

In [ ]:
def data_quality_report(data: pd.DataFrame) -> pd.DataFrame:
    """Resume tipo, nulos, % de nulos y cardinalidad por variable."""
    nulls = data.isnull().sum()
    return pd.DataFrame({
        "Tipo": data.dtypes.astype(str),
        "Nulos": nulls,
        "% Nulos": (nulls / len(data) * 100).round(2),
        "Valores únicos": data.nunique(),
    })


quality = data_quality_report(df)
n_nulls = int(df.isnull().sum().sum())
n_duplicates = int(df.duplicated().sum())

print_header("Validación de calidad · Sección 01")
print(f"Valores nulos totales  : {n_nulls}  ({n_nulls / df.size:.2%})")
print(f"Filas duplicadas       : {n_duplicates}  ({n_duplicates / len(df):.2%})")
check(n_nulls == 0, "0% de valores faltantes")
check(n_duplicates == 0, "0% de registros duplicados")

for id_col in ID_COLS:
    if id_col in df.columns:
        check(df[id_col].is_unique, f"'{id_col}' es un identificador único")

print("\nNulos por variable (.isnull().sum()):")
print(df.isnull().sum().to_string())

quality

In [ ]:
# Consistencia de rangos en variables continuas
hour_cols = [c for c in CONTINUOUS_COLS if c.endswith("Hours") and c in df.columns]

print_header("Consistencia de rangos")
range_table = df[[c for c in CONTINUOUS_COLS if c in df.columns]].agg(["min", "max"]).T
print(range_table.to_string())
print()
for col in hour_cols:
    check(df[col].between(0, 24).all(), f"{col} dentro de [0, 24] horas")

# Combinación de horas diarias que excede 24 h (señal de inconsistencia lógica)
if hour_cols:
    over_24 = (df[hour_cols].sum(axis=1) > 24).sum()
    check(over_24 == 0, f"Registros con suma de horas > 24: {over_24}")

---
## 3. Análisis Exploratorio de Datos (EDA)

### 3.1 Outliers · *Sección 02 — Los extremos también son señal*
Se usa el criterio de Tukey (1,5 × IQR) para cuantificar observaciones extremas en hábitos y bienestar.

In [ ]:
def iqr_outlier_summary(data: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Cuenta outliers por variable según el criterio de Tukey (1.5 * IQR)."""
    rows = []
    for col in columns:
        q1, q3 = data[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((data[col] < lower) | (data[col] > upper)).sum())
        rows.append({
            "Variable": col, "Q1": q1, "Q3": q3, "IQR": iqr,
            "Límite inf.": lower, "Límite sup.": upper,
            "Outliers": n_out, "% Outliers": round(n_out / len(data) * 100, 2),
        })
    return pd.DataFrame(rows).set_index("Variable")


def plot_boxplot_grid(data: pd.DataFrame, columns: list, n_cols: int = 3) -> None:
    """Cuadrícula de diagramas de caja para variables continuas."""
    n_rows = int(np.ceil(len(columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.8 * n_rows))
    for ax, col in zip(axes.flat, columns):
        sns.boxplot(
            x=data[col], ax=ax, color=MT_NAVY, width=0.5,
            boxprops={"alpha": 0.85, "edgecolor": MT_INK},
            medianprops={"color": "white", "linewidth": 2},
            whiskerprops={"color": MT_SLATE}, capprops={"color": MT_SLATE},
            flierprops={"marker": "o", "markersize": 3,
                        "markerfacecolor": MT_ACCENT, "markeredgecolor": MT_ACCENT, "alpha": 0.5},
        )
        style_axis(ax, col.replace("_", " "))
    for ax in list(axes.flat)[len(columns):]:
        ax.set_visible(False)
    fig.suptitle("Evidencia · Boxplot — Outliers conservados para preservar perfiles reales",
                 x=0.01, ha="left", fontsize=14, fontweight="bold", color=MT_NAVY)
    plt.tight_layout()
    plt.show()


plot_boxplot_grid(df, CONTINUOUS_COLS)

outliers = iqr_outlier_summary(df, CONTINUOUS_COLS)
print_header("Resumen de outliers (criterio 1.5 x IQR)")
print(outliers[["Límite inf.", "Límite sup.", "Outliers", "% Outliers"]].to_string())
outliers

In [ ]:
print_header("Decisión metodológica · Tratamiento de outliers")
print("""
Los valores extremos NO se eliminan ni se imputan. Justificación:

  1. Son plausibles: los rangos fueron validados (horas entre 0 y 24, puntajes en escala).
     No son errores de captura, sino comportamientos reales.
  2. Son señal, no ruido: pueden representar privación severa de sueño o uso intensivo
     de redes sociales, justamente los perfiles que MindTrack debe detectar.
  3. Protegen a la clase minoritaria: con solo 6% de casos en riesgo, eliminar extremos
     podría borrar desproporcionadamente a los estudiantes vulnerables.
  4. Trazabilidad: conservar la base original mantiene la reproducibilidad del análisis.

=> Su influencia se controla con estandarización (StandardScaler) y con modelos de
   árboles (Random Forest, XGBoost), robustos a valores extremos.
""")

### 3.2 Composición categórica · *Sección 03 — Una muestra amplia y comparable*
Distribución de `Education_Level`, `Burnout_Level` y `Gender`.

In [ ]:
def annotate_bars(ax, total: int) -> None:
    """Agrega conteo y porcentaje sobre cada barra."""
    for patch in ax.patches:
        height = patch.get_height()
        if height > 0:
            ax.annotate(
                f"{int(height):,}\n({height / total:.1%})".replace(",", "."),
                (patch.get_x() + patch.get_width() / 2, height),
                ha="center", va="bottom", fontsize=9, color=MT_INK,
                xytext=(0, 3), textcoords="offset points",
            )
    ax.margins(y=0.18)


def plot_categorical_grid(data: pd.DataFrame, columns: list, orders: dict = None) -> None:
    """Gráficos de barras para la composición de variables categóricas."""
    orders = orders or {}
    fig, axes = plt.subplots(1, len(columns), figsize=(6 * len(columns), 4.8))
    for ax, col in zip(np.atleast_1d(axes), columns):
        order = orders.get(col, data[col].value_counts().index.tolist())
        order = [o for o in order if o in data[col].unique()]
        colors = sns.blend_palette([MT_NAVY, MT_LIGHT], n_colors=len(order))
        sns.countplot(data=data, x=col, order=order, hue=col, hue_order=order,
                      palette=colors, legend=False, ax=ax, edgecolor=MT_INK, linewidth=0.6)
        annotate_bars(ax, len(data))
        style_axis(ax, col.replace("_", " "), ylabel="Estudiantes")
        ax.tick_params(axis="x", rotation=15)
    fig.suptitle("Evidencia · Composición categórica de la muestra",
                 x=0.01, ha="left", fontsize=14, fontweight="bold", color=MT_NAVY)
    plt.tight_layout()
    plt.show()


CATEGORICAL_EDA = ["Education_Level", ORDINAL_COL, "Gender"]
plot_categorical_grid(df, CATEGORICAL_EDA, orders={ORDINAL_COL: BURNOUT_ORDER})

print_header("Composición categórica")
for col in CATEGORICAL_EDA:
    summary = pd.DataFrame({
        "n": df[col].value_counts(),
        "%": (df[col].value_counts(normalize=True) * 100).round(2),
    })
    print(f"\n>> {col}\n{summary.to_string()}")

### 3.3 Distribuciones univariadas · *Sección 05 — La variabilidad se concentra en zonas legibles*
Histogramas con KDE para leer forma, concentración, colas y asimetrías antes de modelar.
Se complementa con asimetría (*skewness*) y curtosis.

In [ ]:
def plot_hist_kde_grid(data: pd.DataFrame, columns: list, n_cols: int = 3) -> None:
    """Histogramas con curva KDE, media y mediana para variables continuas."""
    n_rows = int(np.ceil(len(columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.8 * n_rows))
    for ax, col in zip(axes.flat, columns):
        sns.histplot(data[col], bins=30, kde=True, ax=ax, color=MT_NAVY,
                     edgecolor="white", alpha=0.75, line_kws={"linewidth": 2})
        ax.lines[0].set_color(MT_INK)
        ax.axvline(data[col].mean(), color=MT_ACCENT, linestyle="--", linewidth=1.5,
                   label=f"Media = {data[col].mean():.2f}")
        ax.axvline(data[col].median(), color=MT_STEEL, linestyle=":", linewidth=1.8,
                   label=f"Mediana = {data[col].median():.2f}")
        ax.legend(fontsize=8, frameon=False)
        style_axis(ax, col.replace("_", " "), ylabel="Frecuencia")
    for ax in list(axes.flat)[len(columns):]:
        ax.set_visible(False)
    fig.suptitle("Evidencia · Histogramas — Distribución de las variables observadas",
                 x=0.01, ha="left", fontsize=14, fontweight="bold", color=MT_NAVY)
    plt.tight_layout()
    plt.show()


plot_hist_kde_grid(df, CONTINUOUS_COLS)

shape_stats = pd.DataFrame({
    "Media": df[CONTINUOUS_COLS].mean(),
    "Mediana": df[CONTINUOUS_COLS].median(),
    "Desv. Est.": df[CONTINUOUS_COLS].std(),
    "Asimetría": df[CONTINUOUS_COLS].skew(),
    "Curtosis": df[CONTINUOUS_COLS].kurt(),
})
shape_stats["Forma"] = np.select(
    [shape_stats["Asimetría"] > 0.5, shape_stats["Asimetría"] < -0.5],
    ["Asimétrica a la derecha", "Asimétrica a la izquierda"],
    default="Aprox. simétrica",
)
print_header("Forma de las distribuciones")
print(shape_stats.round(3).to_string())

#### 3.3.1 Contraste por nivel de riesgo · *Sección 04 — El riesgo emerge al combinar señales*
Se comparan las distribuciones entre estudiantes con y sin riesgo: el grupo en riesgo
**duerme menos, usa más redes sociales y presenta menor salud mental**.

In [ ]:
def plot_distributions_by_target(data: pd.DataFrame, columns: list, target: str,
                                 n_cols: int = 3) -> None:
    """Curvas KDE normalizadas por clase para comparar perfiles de riesgo."""
    n_rows = int(np.ceil(len(columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 3.8 * n_rows))
    for ax, col in zip(axes.flat, columns):
        sns.kdeplot(data=data, x=col, hue=target, common_norm=False, fill=True,
                    alpha=0.35, linewidth=1.8, palette=MT_RISK_PALETTE, ax=ax)
        style_axis(ax, col.replace("_", " "), ylabel="Densidad")
        legend = ax.get_legend()
        if legend is not None:
            legend.set_title("Riesgo")
            legend.set_frame_on(False)
    for ax in list(axes.flat)[len(columns):]:
        ax.set_visible(False)
    fig.suptitle("Evidencia · Contraste de distribuciones por nivel de riesgo",
                 x=0.01, ha="left", fontsize=14, fontweight="bold", color=MT_NAVY)
    plt.tight_layout()
    plt.show()


plot_distributions_by_target(df, CONTINUOUS_COLS, TARGET)

profile = df.groupby(TARGET)[CONTINUOUS_COLS].mean().T
profile.columns = ["Sin riesgo (0)", "En riesgo (1)"]
profile["Diferencia (1-0)"] = profile["En riesgo (1)"] - profile["Sin riesgo (0)"]
print_header("Perfil promedio por clase")
print(profile.round(3).to_string())

### 3.4 Relaciones entre variables · *Sección 06 — Hábitos, bienestar y riesgo están conectados*
Matriz de correlación de Pearson sobre las variables numéricas (incluyendo `Burnout_Level` en su
codificación ordinal). **Valores de referencia del PDF:**
- `Mental_Health_Score` → **r = −0,356** frente al riesgo.
- `Daily_Social_Media_Hours` → **r = +0,330** frente al riesgo.

> La correlación prioriza señales, **no implica causalidad**.

In [ ]:
# Datos numéricos para la correlación (Burnout codificado ordinalmente, sin ID)
corr_df = df.drop(columns=[c for c in ID_COLS if c in df.columns]).copy()
corr_df[ORDINAL_COL] = corr_df[ORDINAL_COL].astype(str).str.strip().str.title().map(BURNOUT_MAP)
corr_df = corr_df.select_dtypes(include="number")

corr_matrix = corr_df.corr(method="pearson")
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 9.5))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".3f", cmap=MT_CMAP, center=0,
            vmin=-1, vmax=1, linewidths=0.6, linecolor="white", square=True,
            annot_kws={"size": 8}, cbar_kws={"shrink": 0.75, "label": "r de Pearson"}, ax=ax)

# Resalta la fila del objetivo
target_idx = corr_matrix.index.get_loc(TARGET)
ax.add_patch(plt.Rectangle((0, target_idx), target_idx + 1, 1, fill=False,
                           edgecolor=MT_ACCENT, linewidth=2.5))
ax.set_title("Evidencia · Matriz de correlación de Pearson", loc="left", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Correlación de cada variable frente al objetivo, ordenada por magnitud
target_corr = (corr_matrix[TARGET].drop(TARGET)
               .sort_values(key=np.abs, ascending=False))

fig, ax = plt.subplots(figsize=(10, 5.5))
colors = [MT_ACCENT if v > 0 else MT_NAVY for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors, edgecolor=MT_INK, linewidth=0.5)
for i, v in enumerate(target_corr.values):
    ax.text(v + (0.008 if v >= 0 else -0.008), i, f"{v:+.3f}", va="center",
            ha="left" if v >= 0 else "right", fontsize=9, color=MT_INK)
ax.axvline(0, color=MT_SLATE, linewidth=1)
ax.invert_yaxis()
ax.margins(x=0.15)
style_axis(ax, f"Correlación de Pearson frente a {TARGET}", xlabel="r")
plt.tight_layout()
plt.show()

print_header(f"Correlación con {TARGET}")
print(target_corr.round(3).to_string())

REFERENCE_CORR = {"Mental_Health_Score": -0.356, "Daily_Social_Media_Hours": 0.330}
print("\nContraste con los valores reportados en la presentación:")
for var, ref in REFERENCE_CORR.items():
    observed = corr_matrix.loc[var, TARGET]
    check(np.isclose(observed, ref, atol=0.005),
          f"{var:<26} r observado = {observed:+.3f} | r PDF = {ref:+.3f}")

leak_corr = corr_matrix.loc["Academic_Performance_Score", TARGET]
print(f"\nNota: Academic_Performance_Score (r = {leak_corr:+.3f}) se excluirá del modelado "
      "por riesgo de data leakage (Sección 08).")

### 3.5 Análisis del desbalance · *Sección 07 — El riesgo se concentra en High o Severe*
**Hallazgos esperados:** 94 % (Clase 0: 14.100) vs 6 % (Clase 1: 900). Los 900 estudiantes en riesgo
se reparten en **450 High y 450 Severe**. La prioridad es maximizar la detección de casos vulnerables.

In [ ]:
class_counts = df[TARGET].value_counts().sort_index()
class_pct = df[TARGET].value_counts(normalize=True).sort_index() * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1, 1.6]})

# (a) Distribución de la variable objetivo
ax = axes[0]
sns.countplot(data=df, x=TARGET, hue=TARGET, palette=MT_RISK_PALETTE, legend=False,
              ax=ax, edgecolor=MT_INK, linewidth=0.6)
annotate_bars(ax, len(df))
ax.set_xticks([0, 1], ["Sin riesgo (0)", "En riesgo (1)"])
style_axis(ax, "Distribución de clases · Academic_Failure_Risk", ylabel="Estudiantes")

# (b) Cruce Burnout_Level x Riesgo
ax = axes[1]
sns.countplot(data=df, x=ORDINAL_COL, hue=TARGET, order=BURNOUT_ORDER,
              palette=MT_RISK_PALETTE, ax=ax, edgecolor=MT_INK, linewidth=0.6)
for container in ax.containers:
    ax.bar_label(container, fmt="{:,.0f}", fontsize=9, padding=2, color=MT_INK)
ax.margins(y=0.12)
ax.legend(title="Riesgo", labels=["Sin riesgo (0)", "En riesgo (1)"], frameon=False)
style_axis(ax, "Burnout y riesgo académico", ylabel="Estudiantes")

fig.suptitle("Evidencia · Burnout y objetivo — distribución de clases",
             x=0.01, ha="left", fontsize=14, fontweight="bold", color=MT_NAVY)
plt.tight_layout()
plt.show()

In [ ]:
print_header("Desbalance de la variable objetivo")
for cls in class_counts.index:
    label = "En riesgo  " if cls == 1 else "Sin riesgo "
    print(f"Clase {cls} ({label}): {class_counts[cls]:>6,}  ->  {class_pct[cls]:5.2f}%"
          .replace(",", "."))
imbalance_ratio = class_counts[0] / class_counts[1]
print(f"\nRatio de desbalance: {imbalance_ratio:.1f} : 1  (por cada estudiante en riesgo hay "
      f"{imbalance_ratio:.1f} sin riesgo)")
check(class_counts.get(0) == 14_100 and class_counts.get(1) == 900,
      "Distribución 14.100 / 900 (94% / 6%)")

# Cruce Burnout x Riesgo
burnout_ct = pd.crosstab(df[ORDINAL_COL], df[TARGET], margins=True, margins_name="Total")
burnout_ct = burnout_ct.reindex(BURNOUT_ORDER + ["Total"])
burnout_ct.columns = ["Sin riesgo (0)", "En riesgo (1)", "Total"]
burnout_ct["% riesgo en el nivel"] = (burnout_ct["En riesgo (1)"] / burnout_ct["Total"] * 100).round(2)

print_header("Cruce Burnout_Level x Academic_Failure_Risk")
print(burnout_ct.to_string())

at_risk_high_severe = burnout_ct.loc[["High", "Severe"], "En riesgo (1)"].sum()
check(at_risk_high_severe == class_counts[1],
      f"El 100% de los casos en riesgo se concentra en High/Severe "
      f"({burnout_ct.loc['High', 'En riesgo (1)']} High + "
      f"{burnout_ct.loc['Severe', 'En riesgo (1)']} Severe)")

print("\nImplicación: la Accuracy es engañosa (un modelo que siempre predice 'sin riesgo' "
      f"obtendría {class_pct[0]:.0f}%).\nSe priorizan Recall, F1 y PR-AUC de la Clase 1.")

### 3.x Gráficas de la presentación (Entregable 2)

Las siguientes visualizaciones replican las diapositivas del segundo corte
(mismos colores, títulos y variables). Los números se validan contra los
valores reportados en el PDF.


In [ ]:
print_header("Gráficas alineadas con diapositivas · Entregable 2")

# Slide 4
plot_target_distribution_slide(df, TARGET)

# Slide 6 — validación de medias globales
for col, (ref_mean, ref_std) in SLIDE_MEANS_REF.items():
    obs_mean, obs_std = df[col].mean(), df[col].std()
    check(np.isclose(obs_mean, ref_mean, atol=0.02), f"{col}: media={obs_mean:.2f} (slide {ref_mean:.2f})")
    if col == "Daily_Social_Media_Hours":
        check(np.isclose(obs_std, ref_std, atol=0.05), f"{col}: DE={obs_std:.2f} (slide {ref_std:.2f})")

# Slide 8
plot_means_by_risk_slide(df, COMPARE_MEAN_COLS, TARGET)

# Slide 10
plot_burnout_risk_percent_slide(df, ORDINAL_COL, TARGET, BURNOUT_ORDER)

# Slide 11
plot_categorical_slide(df, "Gender", "Blues_d", "Distribución por Género")
plot_categorical_slide(df, "Education_Level", "Greens_d", "Distribución por Nivel Educativo")
plot_categorical_slide(df, ORDINAL_COL, "Oranges_d", "Distribución por Nivel de Burnout")

# Slide 9 y 12
plot_histograms_slide(df, HABIT_EDA_COLS)
plot_boxplots_slide(df, HABIT_EDA_COLS)

# Slide 13
corr_slide = plot_correlation_slide(df, ID_COLS, ORDINAL_COL, BURNOUT_MAP, TARGET)
for var, ref in REFERENCE_CORR.items():
    observed = corr_slide.loc[var, TARGET]
    check(np.isclose(observed, ref, atol=0.005), f"{var}: r={observed:+.3f} vs slide {ref:+.3f}")

# Slide 14
plot_risk_boxplots_slide(df, COMPARE_RISK_COLS, TARGET, COMPARE_RISK_TITLES)


---
## 4. Pipeline de preprocesamiento y preparación · *Sección 08 — Control de fugas y transformación*

| Paso | Acción | Justificación |
|---|---|---|
| 1 | Excluir `Academic_Performance_Score` | **Data leakage**: el desempeño académico es prácticamente un resultado del mismo fenómeno que se quiere predecir; no estaría disponible en un escenario preventivo real. |
| 2 | Eliminar `Student_ID` | Identificador sin valor predictivo (riesgo de sobreajuste). |
| 3 | Mapear `Burnout_Level` → 0–3 | Variable **ordinal**: Low (0) < Moderate (1) < High (2) < Severe (3). |
| 4 | One-Hot en `Gender` y `Education_Level` | Variables **nominales** sin orden natural; `drop_first=True` evita colinealidad perfecta. |

> La estandarización (`StandardScaler`) **no** se aplica aquí: se ajusta solo con el conjunto de
> entrenamiento dentro del `Pipeline` del modelo (Bloque 6), evitando fuga de información desde
> Validation/Test.

In [ ]:
def drop_leakage_and_ids(data: pd.DataFrame) -> pd.DataFrame:
    """Paso 1-2: elimina variables con data leakage e identificadores."""
    cols_to_drop = [c for c in LEAKAGE_COLS + ID_COLS if c in data.columns]
    print(f"  - Columnas eliminadas: {cols_to_drop}")
    return data.drop(columns=cols_to_drop)


def encode_ordinal(data: pd.DataFrame, column: str, mapping: dict) -> pd.DataFrame:
    """Paso 3: codificación ordinal con validación de categorías no mapeadas."""
    data = data.copy()
    normalized = data[column].astype(str).str.strip().str.title()
    data[column] = normalized.map(mapping)
    unmapped = normalized[data[column].isna()].unique()
    if len(unmapped) > 0:
        raise ValueError(f"Categorías sin mapear en '{column}': {unmapped}")
    data[column] = data[column].astype(int)
    print(f"  - Mapeo ordinal '{column}': {mapping}")
    return data


def encode_one_hot(data: pd.DataFrame, columns: list, drop_first: bool = True) -> pd.DataFrame:
    """Paso 4: One-Hot Encoding para variables nominales."""
    encoded = pd.get_dummies(data, columns=columns, drop_first=drop_first, dtype=int)
    new_cols = [c for c in encoded.columns if c not in data.columns]
    print(f"  - One-Hot {columns} (drop_first={drop_first}) -> {new_cols}")
    return encoded


def preprocess(data: pd.DataFrame, target: str = TARGET):
    """Ejecuta el pipeline completo y devuelve la matriz X y el vector y."""
    print_header("Pipeline de preprocesamiento · Sección 08")
    processed = (data
                 .pipe(drop_leakage_and_ids)
                 .pipe(encode_ordinal, column=ORDINAL_COL, mapping=BURNOUT_MAP)
                 .pipe(encode_one_hot, columns=NOMINAL_COLS, drop_first=DROP_FIRST_DUMMY))
    X = processed.drop(columns=target)
    y = processed[target].astype(int)
    return X, y


X, y = preprocess(df)

In [ ]:
print_header("Matriz de características final")
print(f"Forma de X : {X.shape[0]:,} registros x {X.shape[1]} características".replace(",", "."))
print(f"Forma de y : {y.shape[0]:,}".replace(",", "."))
print("\nCaracterísticas procesadas:")
for i, (col, dtype) in enumerate(X.dtypes.items(), start=1):
    print(f"  {i:>2}. {col:<35} {str(dtype)}")

print()
n_cols_after_drop = df.shape[1] - len(LEAKAGE_COLS + ID_COLS)
check(n_cols_after_drop == EXPECTED_COLS_AFTER_DROP,
      f"{n_cols_after_drop} variables tras excluir leakage e ID "
      f"({n_cols_after_drop - 1} predictoras + objetivo)")
check(X.shape[1] == EXPECTED_N_FEATURES,
      f"Tras One-Hot (drop_first={DROP_FIRST_DUMMY}) la matriz contiene "
      f"{X.shape[1]} características numéricas")
check(X.isnull().sum().sum() == 0, "Sin valores nulos tras el preprocesamiento")
check(all(pd.api.types.is_numeric_dtype(t) for t in X.dtypes), "Todas las columnas son numéricas")
check(not set(LEAKAGE_COLS + ID_COLS) & set(X.columns), "Sin variables de leakage ni identificadores")

X.head()

---
## 5. Partición estratificada de datos · *Sección 09 — Estratificación representativa*

División en dos etapas con `stratify` y `random_state = 42`:
1. **70 % Train** vs **30 % temporal**.
2. El 30 % temporal se divide a la mitad → **15 % Validation** y **15 % Test**.

| Set | Total | No riesgo / Riesgo |
|---|---|---|
| Train | 10.500 | 9.870 / 630 |
| Validation | 2.250 | 2.115 / 135 |
| Test | 2.250 | 2.115 / 135 |

In [ ]:
def stratified_three_way_split(X: pd.DataFrame, y: pd.Series, train_size: float = 0.70,
                               val_size: float = 0.15, test_size: float = 0.15,
                               random_state: int = RANDOM_STATE):
    """Divide X, y en Train / Validation / Test preservando la proporción de clases."""
    if not np.isclose(train_size + val_size + test_size, 1.0):
        raise ValueError("Las proporciones deben sumar 1.0")

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=val_size + test_size, stratify=y, random_state=random_state
    )
    relative_test = test_size / (val_size + test_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=relative_test, stratify=y_temp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


X_train, X_val, X_test, y_train, y_val, y_test = stratified_three_way_split(X, y)

In [ ]:
def split_summary(splits: dict) -> pd.DataFrame:
    """Tabla resumen de tamaño y tasa de riesgo por partición."""
    rows = []
    for name, target in splits.items():
        rows.append({
            "Set": name,
            "Total": len(target),
            "% del total": len(target) / len(y) * 100,
            "No riesgo (0)": int((target == 0).sum()),
            "Riesgo (1)": int((target == 1).sum()),
            "Tasa de riesgo %": target.mean() * 100,
        })
    return pd.DataFrame(rows).set_index("Set")


splits = {"Train": y_train, "Validation": y_val, "Test": y_test}
summary = split_summary(splits)

print_header("Partición estratificada · Sección 09")
print(f"{'Set':<12}{'Total':>9}{'% total':>10}{'No riesgo':>12}{'Riesgo':>9}{'Tasa riesgo':>14}")
print("-" * 66)
for name, row in summary.iterrows():
    print(f"{name:<12}{row['Total']:>9,.0f}{row['% del total']:>9.1f}%"
          f"{row['No riesgo (0)']:>12,.0f}{row['Riesgo (1)']:>9,.0f}"
          f"{row['Tasa de riesgo %']:>13.2f}%".replace(",", "."))
print("-" * 66)
print(f"{'TOTAL':<12}{summary['Total'].sum():>9,.0f}".replace(",", "."))

print()
EXPECTED_SPLITS = {"Train": (10_500, 630), "Validation": (2_250, 135), "Test": (2_250, 135)}
for name, (total, risk) in EXPECTED_SPLITS.items():
    check(summary.loc[name, "Total"] == total and summary.loc[name, "Riesgo (1)"] == risk,
          f"{name:<10}: {total:,} registros con {risk} casos en riesgo".replace(",", "."))
check(np.allclose(summary["Tasa de riesgo %"], 6.00, atol=0.005),
      "Tasa de riesgo = 6,00% en las 3 particiones")
check(summary["Total"].sum() == len(df), "Sin pérdida ni duplicación de registros")
check(not (set(X_train.index) & set(X_val.index) | set(X_train.index) & set(X_test.index)
           | set(X_val.index) & set(X_test.index)), "Particiones mutuamente excluyentes")

In [ ]:
# Visualización de la partición
fig, ax = plt.subplots(figsize=(10, 3.8))
names = summary.index.tolist()
ax.barh(names, summary["No riesgo (0)"], color=MT_NAVY, label="Sin riesgo (0)")
ax.barh(names, summary["Riesgo (1)"], left=summary["No riesgo (0)"], color=MT_ACCENT,
        label="En riesgo (1)")
for i, (_, row) in enumerate(summary.iterrows()):
    ax.text(row["Total"] + 100, i,
            f"{row['Total']:,.0f}  ·  riesgo {row['Tasa de riesgo %']:.2f}%".replace(",", "."),
            va="center", fontsize=9, color=MT_INK)
ax.invert_yaxis()
ax.set_xlim(0, summary["Total"].max() * 1.35)
ax.legend(frameon=False, loc="lower right")
style_axis(ax, "Partición 70 / 15 / 15 · stratify = y · random_state = 42", xlabel="Registros")
plt.tight_layout()
plt.show()

---
## 6. Configuración de métricas y baseline · *Sección 10 — Detectar a quienes necesitan ayuda*

**Métrica principal:** `Recall (Clase 1) ≥ 0.85` → maximizar la detección y minimizar Falsos Negativos.
**Métricas complementarias:** F1-Score (Clase 1) y PR-AUC (robusta al desbalance).
**Secundarias:** Precision y ROC-AUC.

### 6.1 Función de evaluación estandarizada
Se reutilizará para comparar Regresión Logística, Random Forest y XGBoost en la siguiente fase.

In [ ]:
def evaluate_model(model_name: str, y_true, y_proba, threshold: float = 0.5,
                   recall_target: float = RECALL_TARGET, plot: bool = True) -> dict:
    """
    Evalúa un clasificador binario priorizando el Recall de la Clase 1.

    Parámetros
    ----------
    model_name : nombre descriptivo del modelo.
    y_true     : etiquetas reales (0/1).
    y_proba    : probabilidad estimada de la Clase 1.
    threshold  : umbral de decisión para convertir probabilidades en clases.
    recall_target : Recall mínimo exigido para la Clase 1.
    """
    y_true = np.asarray(y_true)
    y_pred = (np.asarray(y_proba) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "Modelo": model_name,
        "Umbral": threshold,
        "Recall (C1)": recall_score(y_true, y_pred, zero_division=0),
        "Precision (C1)": precision_score(y_true, y_pred, zero_division=0),
        "F1-Score (C1)": f1_score(y_true, y_pred, zero_division=0),
        "PR-AUC": average_precision_score(y_true, y_proba),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "TP": tp, "FN": fn, "FP": fp, "TN": tn,
    }
    metrics["Cumple Recall"] = metrics["Recall (C1)"] >= recall_target

    print_header(f"Evaluación · {model_name} (umbral = {threshold:.3f})")
    status = "CUMPLE" if metrics["Cumple Recall"] else "NO CUMPLE"
    print(f"  >> Recall (Clase 1)   : {metrics['Recall (C1)']:.4f}   "
          f"[objetivo >= {recall_target:.2f} -> {status}]")
    print(f"     F1-Score (Clase 1) : {metrics['F1-Score (C1)']:.4f}")
    print(f"     PR-AUC             : {metrics['PR-AUC']:.4f}   "
          f"(línea base aleatoria = {y_true.mean():.4f})")
    print(f"     Precision (Clase 1): {metrics['Precision (C1)']:.4f}")
    print(f"     ROC-AUC            : {metrics['ROC-AUC']:.4f}")
    print(f"\n  Matriz de confusión: TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}".replace(",", "."))
    print(f"  Estudiantes en riesgo NO detectados (FN): {fn} de {tp + fn}")
    print("\n" + classification_report(y_true, y_pred, target_names=["Sin riesgo", "En riesgo"],
                                       digits=4, zero_division=0))

    if plot:
        _plot_evaluation(model_name, y_true, y_pred, y_proba, metrics)
    return metrics


def _plot_evaluation(model_name, y_true, y_pred, y_proba, metrics) -> None:
    """Matriz de confusión + curva Precision-Recall."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

    cm_cmap = LinearSegmentedColormap.from_list("mt_cm", ["#F8FAFC", MT_NAVY])
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=[0, 1]),
                           display_labels=["Sin riesgo", "En riesgo"]).plot(
        ax=axes[0], cmap=cm_cmap, colorbar=False, values_format=",d")
    axes[0].grid(False)
    axes[0].set_title(f"Matriz de confusión · umbral {metrics['Umbral']:.3f}", loc="left")

    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    axes[1].plot(recall, precision, color=MT_NAVY, linewidth=2,
                 label=f"PR-AUC = {metrics['PR-AUC']:.3f}")
    axes[1].axhline(np.mean(y_true), color=MT_MIST, linestyle="--", label="Clasificador aleatorio")
    axes[1].axvline(RECALL_TARGET, color=MT_ACCENT, linestyle=":", label=f"Recall objetivo {RECALL_TARGET}")
    axes[1].scatter(metrics["Recall (C1)"], metrics["Precision (C1)"], color=MT_ACCENT, s=60,
                    zorder=5, label="Punto de operación")
    axes[1].legend(frameon=False, fontsize=9)
    style_axis(axes[1], "Curva Precision-Recall (Clase 1)", xlabel="Recall", ylabel="Precision")

    fig.suptitle(model_name, x=0.01, ha="left", fontsize=14, fontweight="bold", color=MT_NAVY)
    plt.tight_layout()
    plt.show()

### 6.2 Selección del umbral de decisión
El umbral por defecto (0.5) no garantiza el Recall exigido. Esta función busca, **solo sobre
Validation**, el umbral que alcanza `Recall ≥ 0.85` con el mejor F1 posible. Test permanece intacto
hasta la evaluación final.

In [ ]:
def find_threshold_for_recall(y_true, y_proba, recall_target: float = RECALL_TARGET) -> float:
    """Umbral con mejor F1 entre los que cumplen Recall >= recall_target."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    precision, recall = precision[:-1], recall[:-1]   # alinear con thresholds
    valid = recall >= recall_target
    if not valid.any():
        print("No existe umbral que alcance el Recall objetivo; se usa el mínimo.")
        return float(thresholds.min())
    f1 = 2 * precision[valid] * recall[valid] / (precision[valid] + recall[valid] + 1e-12)
    return float(thresholds[valid][np.argmax(f1)])

### 6.3 Baseline · Regresión Logística · *Modelos — La línea base interpretable del riesgo*
- `StandardScaler` dentro del `Pipeline` → se ajusta únicamente con Train.
- `class_weight="balanced"` → compensa el desbalance 94/6 penalizando más los errores en la Clase 1.

In [ ]:
baseline_lr = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1_000,
                                 random_state=RANDOM_STATE)),
])
baseline_lr.fit(X_train, y_train)

proba_val = baseline_lr.predict_proba(X_val)[:, 1]
print_header("Baseline entrenado")
print(f"Entrenado con {len(X_train):,} registros y {X_train.shape[1]} características."
      .replace(",", "."))

In [ ]:
# (a) Umbral por defecto 0.5
metrics_default = evaluate_model("Regresión Logística · Validation", y_val, proba_val, threshold=0.5)

In [ ]:
# (b) Umbral ajustado en Validation para cumplir Recall >= 0.85
tuned_threshold = find_threshold_for_recall(y_val, proba_val, RECALL_TARGET)
metrics_tuned = evaluate_model("Regresión Logística · Validation (umbral ajustado)",
                               y_val, proba_val, threshold=tuned_threshold)

### 6.4 Interpretación del baseline: coeficientes estandarizados

In [ ]:
coefs = pd.Series(baseline_lr.named_steps["model"].coef_[0], index=X_train.columns)
coefs = coefs.sort_values(key=np.abs, ascending=True)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(coefs.index, coefs.values,
        color=[MT_ACCENT if v > 0 else MT_NAVY for v in coefs.values],
        edgecolor=MT_INK, linewidth=0.5)
ax.axvline(0, color=MT_SLATE, linewidth=1)
style_axis(ax, "Regresión Logística · coeficientes (variables estandarizadas)",
           xlabel="Coeficiente (log-odds)  ·  ámbar = aumenta el riesgo, azul = lo reduce")
plt.tight_layout()
plt.show()

odds = pd.DataFrame({"Coeficiente": coefs, "Odds ratio (por 1 DE)": np.exp(coefs)})
print_header("Coeficientes del baseline")
print(odds.sort_values("Coeficiente", key=np.abs, ascending=False).round(3).to_string())

### 6.5 Resumen comparativo del baseline

In [ ]:
results = pd.DataFrame([metrics_default, metrics_tuned]).set_index("Modelo")
metric_cols = ["Umbral", "Recall (C1)", "Precision (C1)", "F1-Score (C1)", "PR-AUC", "ROC-AUC",
               "FN", "FP", "Cumple Recall"]

print_header("Resumen del baseline (Validation)")
print(results[metric_cols].round(4).to_string())
print("\nEl conjunto de Test (2.250 registros) se mantiene reservado para la evaluación final "
      "y comparativa\nde Regresión Logística, Random Forest y XGBoost.")
results[metric_cols]

---
## Conclusiones y próximos pasos

**01 · Estado actual — Base confiable:** datos validados (0 % nulos, 0 % duplicados), explorados,
preprocesados (sin data leakage) y estratificados (6,00 % de riesgo en cada partición). Tras excluir
leakage e ID quedan **12 variables** (11 predictoras + objetivo), que tras el One-Hot Encoding forman una
matriz de **13 características numéricas**.

**02 · Regresión Logística en este notebook:** únicamente **línea base** (umbral y Recall en Validation).
**No** se entrenan Random Forest ni XGBoost aquí.

**03 · Siguiente notebook (aparte):** comparación de los **tres modelos**, gráficas comparativas y modelo ganador
→ [MindTrack_03_Comparacion_Modelos.ipynb · Entregable 3](https://colab.research.google.com/gist/Danielhsk/92fab1e431edaeb0ce998bb03a09d04c/MindTrack_03_Comparacion_Modelos.ipynb).

**04 · Entregable final:** prototipo Streamlit + SHAP + recomendaciones.

> *Roadmap:* **Entregable 2 (datos + baseline) → Entregable 3 (comparar modelos) → prototipo web.*
> Criterio de éxito: **Recall (C1) ≥ 0.85**.
